# 📘 Notebook 1: Univariate & Multivariate Calculus — Partial Derivatives & Jacobians

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐ (Beginner-Friendly)

---

## 🏠 Why Does This Matter? (Connect to ML First)

Imagine you are building a model to predict **house prices**.

Your model says:  
> predicted price = w₁ × (square footage) + w₂ × (bedrooms) + b

After training, your model is off. The real price was \$400k, your model said \$350k.  
You need to fix `w₁` and `w₂`. But **how?** Which direction do you adjust them?

**Calculus answers this question.** It tells you:
- If you increase `w₁` by a tiny amount, does the error get bigger or smaller?
- By how much?

That rate-of-change is called a **derivative** — and it is the mathematical engine behind every ML model that learns.

---

## 📚 What You Need to Know Beforehand
- Python basics (functions, loops, numpy arrays)
- What a matrix multiplication is (from Week 1)
- No prior calculus required — we build it from scratch!

---

## Part 1: What is a Derivative? (Univariate Calculus)

### Plain English First

Imagine you are driving a car. Your speedometer reads **60 mph**.  
That speed is the **derivative** of your position with respect to time:  
> "How fast is my position changing right now?"

In the same way, for any function `f(x)`:  
> **Derivative = How fast does the OUTPUT change when the INPUT changes by a tiny amount?**

### The Mathematical Definition

Plain English first: "nudge the input by a tiny h, measure how much the output changes, divide by h".

$$f'(x) = \frac{df}{dx} = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

### House Price Analogy

Let `L(w)` = the prediction error (loss) when weight = `w`.  
The derivative `dL/dw` tells you:  
- **Positive** → increasing `w` makes the error *bigger* → we should *decrease* `w`  
- **Negative** → increasing `w` makes the error *smaller* → we should *increase* `w`  
- **Zero** → we are at the minimum — no adjustment needed!

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SETUP: import all libraries we need
# ─────────────────────────────────────────────────────────────────
import numpy as np                  # numerical computing
import matplotlib.pyplot as plt     # plotting
from mpl_toolkits.mplot3d import Axes3D  # 3D plots

In [ ]:
# ─────────────────────────────────────────────────────────────────
# EXAMPLE: A simple loss function L(w) = w²
#
# Think of it as: if our weight w is far from 0, the error is large.
# The minimum error is 0 (achieved when w = 0).
# ─────────────────────────────────────────────────────────────────

# Create 300 evenly-spaced weight values from -3 to 3
w = np.linspace(-3, 3, 300)

L     = w**2         # the loss function L(w) = w²
dL_dw = 2 * w        # the derivative dL/dw = 2w  (computed analytically)

# ─── Plot the function and its derivative side by side ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: the loss function
axes[0].plot(w, L, 'steelblue', linewidth=2.5, label='L(w) = w²')
axes[0].plot(0, 0, 'ro', markersize=12, label='Minimum (w=0, L=0)')

# Add tangent lines at two points to show the slope visually
for point, color in [(-2, 'green'), (1.5, 'orange')]:
    slope = 2 * point                          # derivative at this point
    x_tang = np.linspace(point - 0.8, point + 0.8, 50)
    y_tang = point**2 + slope * (x_tang - point)  # tangent line formula
    axes[0].plot(x_tang, y_tang, color=color, linestyle='--', linewidth=2,
                 label=f'Tangent at w={point} (slope={slope})')

axes[0].set_title('Loss Function L(w) = w²', fontsize=13)
axes[0].set_xlabel('w  (weight value)')
axes[0].set_ylabel('L(w)  (loss / error)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0, color='k', linewidth=0.5)
axes[0].axvline(0, color='k', linewidth=0.5)

# Right plot: the derivative (slope at each point)
axes[1].plot(w, dL_dw, 'crimson', linewidth=2.5, label="dL/dw = 2w")
axes[1].axhline(0, color='k', linestyle='--', linewidth=1, label='Zero slope (minimum)')
axes[1].plot(0, 0, 'go', markersize=12, zorder=5, label='w=0: slope=0 → MINIMUM')
axes[1].fill_between(w, dL_dw, 0,
                     where=(dL_dw > 0), alpha=0.15, color='red',
                     label='Positive slope → decrease w')
axes[1].fill_between(w, dL_dw, 0,
                     where=(dL_dw < 0), alpha=0.15, color='blue',
                     label='Negative slope → increase w')
axes[1].set_title("Derivative dL/dw = 2w", fontsize=13)
axes[1].set_xlabel('w  (weight value)')
axes[1].set_ylabel('dL/dw  (rate of change)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Derivative = The Slope of the Loss at Each Point', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key insight:")
print("  At w = -2:  slope = -4  → error decreases as w increases → move w RIGHT")
print("  At w =  0:  slope =  0  → at the MINIMUM, no movement needed")
print("  At w =  2:  slope = +4  → error increases as w increases → move w LEFT")

## Part 2: Computing Derivatives Without Calculus — Finite Differences

### Plain English First

Computers can't do the "limit as h→0" exactly, but they can use a very small `h`.

The idea: **nudge the input by a tiny ε, see how much the output moves, divide by ε.**

$$\frac{df}{dx} \approx \frac{f(x + \varepsilon) - f(x - \varepsilon)}{2\varepsilon} \quad \text{for small } \varepsilon \text{ (e.g., } 10^{-5}\text{)}$$

This is called the **finite difference approximation** — it is how ML frameworks verify their gradient computations.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Numerical derivative using finite differences
# This is the "gradient check" technique used in neural network debugging
# ─────────────────────────────────────────────────────────────────

def numerical_derivative(func, x, eps=1e-5):
    """
    Approximate df/dx at point x using central finite differences.

    Why central (using x+eps AND x-eps)?  It is more accurate than
    one-sided (just x+eps), because errors cancel out symmetrically.

    func : the function to differentiate
    x    : the point at which to compute the derivative
    eps  : the tiny nudge (1e-5 is standard for gradient checking)
    """
    return (func(x + eps) - func(x - eps)) / (2 * eps)   # central difference formula


# ─── Test on several functions, comparing numerical vs analytical ──
# We know the analytical answers from calculus rules:
#   d/dx [x²]      = 2x
#   d/dx [x³]      = 3x²
#   d/dx [sin(x)]  = cos(x)
#   d/dx [e^x]     = e^x

test_cases = [
    ("L(w) = w²",     lambda w: w**2,        lambda w: 2*w,           2.0),
    ("L(w) = w³",     lambda w: w**3,        lambda w: 3*w**2,        2.0),
    ("L(w) = sin(w)", lambda w: np.sin(w),   lambda w: np.cos(w),     1.0),
    ("L(w) = e^w",    lambda w: np.exp(w),   lambda w: np.exp(w),     0.5),
]

print(f"{'Function':20} | {'Point':6} | {'Numerical':15} | {'Analytical':15} | {'Error':12}")
print("-" * 75)
for name, func, analytic, x_test in test_cases:
    num  = numerical_derivative(func, x_test)   # our approximation
    true = analytic(x_test)                      # the exact answer
    err  = abs(num - true)                        # how wrong are we?
    print(f"{name:20} | {x_test:6.2f} | {num:15.10f} | {true:15.10f} | {err:.2e}")

print()
print("The error is about 1e-10 — almost perfect!")
print("This is the GRADIENT CHECKING technique used to debug backpropagation.")

## Part 3: Multivariate Calculus — When Multiple Weights Affect the Loss

### Plain English First

Our house price model has **two weights**: `w₁` (for square footage) and `w₂` (for bedrooms).  
The loss `L(w₁, w₂)` depends on **both** simultaneously.

Imagine a mountainous landscape where:  
- The `x`-axis is `w₁`, the `y`-axis is `w₂`, the `z`-axis is the loss
- You want to find the **valley** (minimum loss)
- To do that, you need to know the slope in **each direction separately**

### What is a Partial Derivative?

Plain English: **"Hold everything else fixed. Wiggle just this one variable. How much does L change?"**

$$\frac{\partial L}{\partial w_1} = \text{slope in the } w_1 \text{ direction (treat } w_2 \text{ as a constant)}$$

$$\frac{\partial L}{\partial w_2} = \text{slope in the } w_2 \text{ direction (treat } w_1 \text{ as a constant)}$$

**Worked example:**  
$$L(w_1, w_2) = w_1^2 + 3w_1 w_2 + w_2^2$$

$$\frac{\partial L}{\partial w_1} = 2w_1 + 3w_2 \quad \text{(treat } w_2 \text{ as a number, differentiate normally)}$$

$$\frac{\partial L}{\partial w_2} = 3w_1 + 2w_2 \quad \text{(treat } w_1 \text{ as a number)}$$

In [ ]:
# ─────────────────────────────────────────────────────────────────
# HOUSE PRICE EXAMPLE
#
# Suppose our prediction error (loss) simplifies to:
#   L(w1, w2) = (w1 - 2)² + (w2 + 1)²
#
# This means:
#   - The best w1 is 2.0  (minimizes the w1 part)
#   - The best w2 is -1.0 (minimizes the w2 part)
#
# Partial derivatives:
#   ∂L/∂w1 = 2(w1 - 2)    → tells us how to adjust w1
#   ∂L/∂w2 = 2(w2 + 1)    → tells us how to adjust w2
# ─────────────────────────────────────────────────────────────────

def loss(w1, w2):
    """Simplified house price loss. True optimum: w1=2, w2=-1."""
    return (w1 - 2)**2 + (w2 + 1)**2

def dL_dw1(w1, w2):
    """Partial derivative of L with respect to w1. w2 treated as constant."""
    return 2 * (w1 - 2)

def dL_dw2(w1, w2):
    """Partial derivative of L with respect to w2. w1 treated as constant."""
    return 2 * (w2 + 1)


# ─── Create a grid of (w1, w2) values to plot the loss surface ────
w1_vals = np.linspace(-1, 5, 200)    # weight 1 range
w2_vals = np.linspace(-4, 2, 200)    # weight 2 range
W1, W2  = np.meshgrid(w1_vals, w2_vals)  # create 2D grid
Z       = loss(W1, W2)               # compute loss at every grid point


# ─── Plot: 3D surface + contour map with partial derivative info ──
fig = plt.figure(figsize=(16, 6))

# 3D surface: the loss landscape
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(W1, W2, Z, cmap='plasma', alpha=0.85, rstride=5, cstride=5)
ax1.scatter([2], [-1], [0], color='lime', s=100, zorder=10)  # the minimum
ax1.set_xlabel('w₁ (sqft weight)')
ax1.set_ylabel('w₂ (bedroom weight)')
ax1.set_zlabel('Loss')
ax1.set_title('Loss Surface\n(bowl shape → one minimum)', fontsize=11)

# Contour map with ∂L/∂w1 shown
ax2 = fig.add_subplot(132)
c2  = ax2.contourf(W1, W2, 2*(W1-2), levels=20, cmap='RdBu_r')  # = ∂L/∂w1
plt.colorbar(c2, ax=ax2, label='Value of ∂L/∂w1')
ax2.plot(2, -1, 'g*', markersize=15, label='Optimum (w1=2, w2=-1)')
ax2.axvline(2, color='lime', linestyle='--', linewidth=1.5, label='∂L/∂w1 = 0 line')
ax2.set_title('∂L/∂w1 = 2(w1−2)\nRed=positive (move w1 left)\nBlue=negative (move w1 right)', fontsize=10)
ax2.set_xlabel('w₁'); ax2.set_ylabel('w₂'); ax2.legend(fontsize=8)

# Contour map with ∂L/∂w2 shown
ax3 = fig.add_subplot(133)
c3  = ax3.contourf(W1, W2, 2*(W2+1), levels=20, cmap='RdBu_r')  # = ∂L/∂w2
plt.colorbar(c3, ax=ax3, label='Value of ∂L/∂w2')
ax3.plot(2, -1, 'g*', markersize=15, label='Optimum')
ax3.axhline(-1, color='lime', linestyle='--', linewidth=1.5, label='∂L/∂w2 = 0 line')
ax3.set_title('∂L/∂w2 = 2(w2+1)\nRed=positive (move w2 down)\nBlue=negative (move w2 up)', fontsize=10)
ax3.set_xlabel('w₁'); ax3.set_ylabel('w₂'); ax3.legend(fontsize=8)

plt.suptitle('House Price Model Loss: L(w1,w2) = (w1−2)² + (w2+1)²\nPartial derivatives guide us to the minimum', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Verify partial derivatives using the numerical approach
# ─────────────────────────────────────────────────────────────────

def partial_deriv_numerical(func, params, which_index, eps=1e-5):
    """
    Compute ∂func/∂params[which_index] numerically.

    params       : list [w1, w2, w3, ...]
    which_index  : 0 for w1, 1 for w2, etc.
    eps          : the tiny nudge

    The trick: only nudge the ONE variable we are differentiating w.r.t.
    """
    params_plus  = list(params)                   # make a copy
    params_minus = list(params)                   # make another copy
    params_plus[which_index]  += eps              # nudge one variable up
    params_minus[which_index] -= eps              # nudge one variable down
    return (func(*params_plus) - func(*params_minus)) / (2 * eps)  # central difference


# Test at several (w1, w2) points
test_pts = [(0.0, 0.0), (1.5, -0.5), (2.0, -1.0), (3.0, 1.0)]

print("Verifying partial derivatives of L(w1,w2) = (w1-2)² + (w2+1)²")
print(f"{'(w1, w2)':15} | {'∂L/∂w1 analytic':18} | {'∂L/∂w1 numeric':18} | {'∂L/∂w2 analytic':18} | {'∂L/∂w2 numeric':18}")
print("-" * 93)
for (w1v, w2v) in test_pts:
    # Analytical values
    anal_w1 = dL_dw1(w1v, w2v)
    anal_w2 = dL_dw2(w1v, w2v)
    # Numerical values
    num_w1  = partial_deriv_numerical(loss, [w1v, w2v], which_index=0)
    num_w2  = partial_deriv_numerical(loss, [w1v, w2v], which_index=1)
    print(f"({w1v:.1f}, {w2v:.1f}):        | {anal_w1:18.8f} | {num_w1:18.8f} | {anal_w2:18.8f} | {num_w2:18.8f}")

print()
print("At (2.0, -1.0): both partial derivatives = 0.  This is the MINIMUM — exactly where we want to be!")

## Part 4: The Jacobian — When the Model Has Multiple Outputs

### Plain English First

So far: one output (house price), one set of partial derivatives.  
But what if your model predicts **multiple things** simultaneously?

Example: predict both [house price, rental income] from features [sqft, bedrooms].  
Now every output depends on every input, and we need **a full table of derivatives**.

That table is called the **Jacobian matrix**.

$$J = \begin{bmatrix}
\frac{\partial \text{price}}{\partial \text{sqft}} & \frac{\partial \text{price}}{\partial \text{beds}} \\
\frac{\partial \text{rent}}{\partial \text{sqft}}  & \frac{\partial \text{rent}}{\partial \text{beds}}
\end{bmatrix}$$

**Row i = derivatives of output i. Column j = derivatives w.r.t. input j.**

In a neural network layer `h = W·x`:  
The Jacobian `∂h/∂x = W` — this is exactly **why weight matrices appear in backpropagation!**

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Computing the Jacobian numerically
#
# Example: A neural network linear layer: h = W @ x
#   W is (3 outputs × 2 inputs)
#   x is a 2D input (sqft_normalized, bedrooms_normalized)
#   h is a 3D output (three hidden units)
#
# The Jacobian ∂h/∂x should equal W exactly!
# ─────────────────────────────────────────────────────────────────

def compute_jacobian(func, x, eps=1e-5):
    """
    Compute the Jacobian matrix of func at point x.

    func : takes a vector x of length n, returns a vector of length m
    x    : input point, shape (n,)
    Returns J of shape (m, n) — m outputs, n inputs.

    Each column j of J is the partial derivative of ALL outputs
    with respect to input j.
    """
    x    = np.array(x, dtype=float)    # ensure float for arithmetic
    f0   = np.array(func(x))           # evaluate function at x → shape (m,)
    m    = len(f0)                     # number of outputs
    n    = len(x)                      # number of inputs
    J    = np.zeros((m, n))            # Jacobian starts as all zeros

    for j in range(n):                 # loop over each input dimension
        x_plus  = x.copy(); x_plus[j]  += eps    # nudge input j up
        x_minus = x.copy(); x_minus[j] -= eps    # nudge input j down
        J[:, j] = (func(x_plus) - func(x_minus)) / (2 * eps)  # fill column j
    return J


# Define a linear layer  h = W @ x
np.random.seed(42)
W_layer = np.array([[1.0, 2.0],    # row 0: hidden unit 0
                    [3.0, 4.0],    # row 1: hidden unit 1
                    [0.5, -1.0]])  # row 2: hidden unit 2
# W is shape (3, 2) — 3 outputs, 2 inputs

def linear_layer(x):
    """A simple linear transformation: output = W @ x."""
    return W_layer @ x

x_test = np.array([1.0, 2.0])   # sqft_norm=1, bedrooms_norm=2

# Compute Jacobian numerically
J_numerical  = compute_jacobian(linear_layer, x_test)

print("Linear layer h = W @ x")
print(f"W (the weight matrix) = ")
print(W_layer)
print(f"\nNumerical Jacobian ∂h/∂x:")
print(J_numerical)
print(f"\nMax difference (should be ~0): {np.max(np.abs(J_numerical - W_layer)):.2e}")
print()
print("KEY INSIGHT: The Jacobian of a linear layer is exactly the weight matrix W!")
print("This explains why W.T appears in the backpropagation formula: ∂L/∂x = W.T @ ∂L/∂h")

## Part 5: Common Derivative Rules (Quick Reference)

You don't need to memorize all these — just recognize the patterns when you see them.

| Rule | Formula | Example | Why it matters in ML |
|------|---------|---------|---------------------|
| **Power** | d/dx[xⁿ] = nxⁿ⁻¹ | d/dx[x²] = 2x | Loss = MSE = (pred-true)² |
| **Sum** | d/dx[f+g] = f'+g' | d/dx[x²+x] = 2x+1 | Loss = sum of terms |
| **Chain** | d/dx[f(g)] = f'(g)·g' | d/dx[sin(x²)] = cos(x²)·2x | Backprop through layers |
| **exp** | d/dx[eˣ] = eˣ | — | Sigmoid, softmax activation |
| **log** | d/dx[ln x] = 1/x | — | Cross-entropy loss |
| **constant** | d/dx[c] = 0 | d/dx[5] = 0 | Biases in layers |

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Verify all derivative rules numerically — a useful debugging habit
# ─────────────────────────────────────────────────────────────────

def check_derivative(name, func, analytic_deriv, x_test, eps=1e-7):
    """
    Compare an analytically-derived derivative to the numerical one.
    This is exactly what PyTorch's 'gradcheck' does internally.
    """
    numerical  = (func(x_test + eps) - func(x_test - eps)) / (2 * eps)
    analytical = analytic_deriv(x_test)
    error      = abs(numerical - analytical)
    status     = '✓' if error < 1e-6 else '✗'
    print(f"  {status} {name:30}:  analytical={analytical:10.6f},  numerical={numerical:10.6f},  error={error:.2e}")

x = 1.5   # test at x=1.5
print(f"Checking derivative rules at x = {x}:")
print()

# (function, analytical derivative)
checks = [
    ("Power:   d/dx[x³]",        lambda x: x**3,             lambda x: 3*x**2),
    ("Sum:     d/dx[x²+3x]",     lambda x: x**2 + 3*x,       lambda x: 2*x + 3),
    ("Exp:     d/dx[e^x]",       lambda x: np.exp(x),        lambda x: np.exp(x)),
    ("Log:     d/dx[ln(x)]",     lambda x: np.log(x),        lambda x: 1/x),
    ("Sigmoid: d/dx[1/(1+e^-x)]",lambda x: 1/(1+np.exp(-x)), lambda x: (1/(1+np.exp(-x)))*(1-1/(1+np.exp(-x)))),
    ("Chain:   d/dx[sin(x²)]",   lambda x: np.sin(x**2),     lambda x: np.cos(x**2)*2*x),
]

for name, func, deriv in checks:
    check_derivative(name, func, deriv, x)

---

## ✅ Self-Check Questions

Test your understanding before moving on:

**1.** The loss of your house price model at weight `w=3` is `L=9`, and `dL/dw = 6`.  
   Should you increase or decrease `w` to reduce the loss? Why?

**2.** If `L(w1, w2) = w1² + 2w2²`, what is `∂L/∂w1` and `∂L/∂w2`?  
   (Hint: when finding `∂L/∂w1`, treat `w2` as a constant.)

**3.** At which point (w1, w2) are both partial derivatives equal to zero?  
   What does that point represent?

**4.** The Jacobian of a linear layer `h = W @ x` is equal to ___?

**5.** Why do we use central finite differences `[f(x+ε)-f(x-ε)]/(2ε)` instead of  
   one-sided `[f(x+ε)-f(x)]/ε`?

<details>
<summary>Click to see answers</summary>

1. **Decrease w** — the derivative is positive (+6), meaning increasing `w` increases the loss. To reduce the loss, move w in the opposite direction.  

2. `∂L/∂w1 = 2w1` (treat w2 as constant, differentiate w1² normally).  
   `∂L/∂w2 = 4w2` (treat w1 as constant, differentiate 2w2² → 4w2).  

3. Both are zero at **(0, 0)**. This is the **minimum** of the loss function — the optimal weights.  

4. The Jacobian equals **W** (the weight matrix itself).  

5. Central differences are more accurate — the error is O(ε²) instead of O(ε), so we get ~10 more digits of precision for the same ε.

</details>

---

## 📌 Summary

| Concept | One-line meaning | House price analogy |
|---------|-----------------|---------------------|
| **Derivative** | Rate of change of output w.r.t. input | How does loss change when I adjust weight w? |
| **Partial derivative** | Rate of change w.r.t. one variable (others fixed) | How does loss change when I adjust w₁ only? |
| **Finite difference** | Numerical approximation of derivative | Nudge the weight, measure change in loss |
| **Jacobian** | Matrix of all partial derivatives | All sensitivities across all layers |

**➡️ Next Notebook:** Chain Rule & Backpropagation — how partial derivatives chain together across many layers to train a neural network.